# Transfer Learning (Keras) — Pretrained VGG16 on CIFAR-10

**Added later** to fill a gap: this notebook was an empty stub in the original coursework (see the folder README). Written now in Keras, consistent with the other from-scratch Keras models in this folder (`LeNet_5_Keras`, `ResNet_Keras`, `VGG16_Keras`) — but here the point is the *opposite* of building the network from scratch: start from ImageNet-pretrained weights and fine-tune. Not executed in this environment — no `tensorflow`/`keras` installed here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.datasets import cifar10
from keras.applications import VGG16
from keras.layers import Dense, Flatten, GlobalAveragePooling2D
from keras.models import Model
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from keras.utils import to_categorical


In [ ]:
(trainX, trainY), (testX, testY) = cifar10.load_data()

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

trainX = trainX.astype('float32') / 255.0
testX = testX.astype('float32') / 255.0

trainY_cat = to_categorical(trainY, 10)
testY_cat = to_categorical(testY, 10)

print('trainX shape:', trainX.shape, ' testX shape:', testX.shape)


In [ ]:
plt.figure(figsize=(6, 6))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(trainX[i])
    plt.title(classes[trainY[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()


## Stage 1 — Feature extraction

Load `VGG16` with ImageNet weights, drop the classifier head, freeze the convolutional
base, and train only a new small classifier head on CIFAR-10. VGG16's `include_top=False`
mode accepts inputs as small as 32x32, which matches CIFAR-10 natively (no upscaling
needed).

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False

model = Model(
    inputs=base_model.input,
    outputs=Dense(10, activation='softmax')(
        Dense(256, activation='relu')(
            GlobalAveragePooling2D()(base_model.output)
        )
    ),
)

model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
datagen.fit(trainX)

history_head = model.fit(
    datagen.flow(trainX, trainY_cat, batch_size=64),
    validation_data=(testX, testY_cat),
    epochs=10,
)


## Stage 2 — Fine-tuning

Unfreeze the last convolutional block of VGG16 and continue training at a much lower
learning rate, so the pretrained low-level filters aren't destroyed by large gradient
updates.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history_fine_tune = model.fit(
    datagen.flow(trainX, trainY_cat, batch_size=64),
    validation_data=(testX, testY_cat),
    epochs=10,
)


In [ ]:
def plot_history(*histories):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history['accuracy']
        val_acc += h.history['val_accuracy']
        loss += h.history['loss']
        val_loss += h.history['val_loss']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(acc, label='train'); axes[0].plot(val_acc, label='val')
    axes[0].set_title('Accuracy'); axes[0].legend()
    axes[1].plot(loss, label='train'); axes[1].plot(val_loss, label='val')
    axes[1].set_title('Loss'); axes[1].legend()
    plt.show()

plot_history(history_head, history_fine_tune)


In [ ]:
test_loss, test_acc = model.evaluate(testX, testY_cat)
print(f'Test accuracy after fine-tuning: {test_acc:.4f}')
